In [ ]:
!pip install -q transformers torch faiss-cpu Pillow tqdm h5py

In [ ]:
import os, time, numpy as np, torch, torch.nn.functional as F
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoImageProcessor
import faiss, h5py, pickle

# Universal secret getter - works on Kaggle (Secrets or Private Dataset) and local (.env)
def get_secret(key: str, fallback_path: str = None, default=None) -> str:
    # Try Kaggle Secrets first (manual run)
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(key)
    except Exception:
        pass
    
    # Try private dataset fallback (API push)
    if fallback_path and os.path.exists(fallback_path):
        return open(fallback_path).read().strip()
    
    # Try .env file (local dev)
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    
    value = os.environ.get(key, default)
    if value is None:
        raise EnvironmentError(f"Secret '{key}' not found in Kaggle Secrets, private dataset, or .env")
    return value

# Get HF_TOKEN (3-tier fallback)
hf_token = get_secret("HF_TOKEN", fallback_path="/kaggle/input/my-hf-secrets/HF_TOKEN.txt")
print("✓ HF_TOKEN loaded")

# Detect dataset path
paths = [
    "/kaggle/input/chetankv/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/dogs-cats-images/dataset/test_set/cats",
    "/kaggle/input/datasets/chetankv/dogs-cats-images/dataset/test_set/cats"
]
DATASET_PATH = next((p for p in paths if os.path.exists(p)), None)
if not DATASET_PATH: raise FileNotFoundError("Dataset not found")

MODEL_ID = "facebook/dinov3-vitb16-pretrain-lvd1689m"
EMBEDDING_DIM = 768
OUTPUT_DIR = "/kaggle/working"

# ADAPTIVE CONFIGURATION
gpu_name = ""
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    
    # P100 not supported by current PyTorch - fallback to CPU
    if "P100" in gpu_name:
        device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
        print(f"GPU: {gpu_name} (not supported by PyTorch) - using CPU mode")
    elif num_gpus > 1:
        device = "cuda"
        BATCH_SIZE = 256  # 32GB Total VRAM (T4x2)
        NUM_WORKERS = 4
        print(f"GPU: {gpu_name} | Count: {num_gpus} | Batch: {BATCH_SIZE}")
    else:
        device = "cuda"
        BATCH_SIZE = 192  # T4 single
        NUM_WORKERS = 2
        print(f"GPU: {gpu_name} | Count: {num_gpus} | Batch: {BATCH_SIZE}")
else:
    device, BATCH_SIZE, NUM_WORKERS = "cpu", 32, 2
    print("CPU mode")


In [ ]:
class FastDataset(Dataset):
    def __init__(self, paths, proc):
        self.paths, self.proc = paths, proc
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except:
            img = Image.new("RGB", (224,224), 0)
            
        inputs = self.proc(images=img, return_tensors="pt")
        inputs_squeezed = {k: v.squeeze(0) for k, v in inputs.items()}
        return inputs_squeezed, self.paths[i]

imgs = [(str(p.relative_to(DATASET_PATH)), str(p)) for p in Path(DATASET_PATH).rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp"}]
image_ids, image_paths = [i for i,_ in imgs], [p for _,p in imgs]


In [ ]:
print(f"Loading {MODEL_ID}...")
t0 = time.time()
processor = AutoImageProcessor.from_pretrained(MODEL_ID, token=hf_token)

dtype = torch.float16 if device == "cuda" else torch.float32
use_amp = True if device == "cuda" else False
if use_amp: print("  Using FP16 for speedup.")

model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=dtype, low_cpu_mem_usage=True, token=hf_token)

# DINOv3 doesn't have text model, so no surgery needed
torch.cuda.empty_cache()

class VisionWrapper(torch.nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.model = full_model
        
    def forward(self, *args, **kwargs):
        outputs = self.model(*args, **kwargs)
        # DINOv3 returns pooler_output directly
        return outputs.pooler_output if hasattr(outputs, 'pooler_output') else outputs.last_hidden_state[:, 0]

wrapped_model = VisionWrapper(model)

if device == "cuda" and torch.cuda.device_count() > 1:
    final_model = torch.nn.DataParallel(wrapped_model).cuda()
    print("  ✓ Applied DataParallel")
else:
    final_model = wrapped_model.to(device)

final_model.eval()
if device == "cuda": torch.backends.cudnn.benchmark = True
print(f"Model ready in {time.time()-t0:.1f}s")


In [ ]:
loader = DataLoader(FastDataset(image_paths, processor), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(device=="cuda"), persistent_workers=(NUM_WORKERS>0), prefetch_factor=2 if NUM_WORKERS > 0 else None)
all_emb, t0 = [], time.time()

with torch.no_grad():
    for inputs, _ in tqdm(loader, desc="Inference"):
        inputs = {k: v.to(device, non_blocking=True) for k, v in inputs.items()}
        
        if use_amp:
            with torch.amp.autocast('cuda', dtype=torch.float16):
                pooler_output = final_model(**inputs)
                feat = F.normalize(pooler_output, p=2, dim=-1)
        else:
            pooler_output = final_model(**inputs)
            feat = F.normalize(pooler_output, p=2, dim=-1)
            
        all_emb.append(feat.cpu().numpy())

if device == "cuda": torch.cuda.synchronize()
inf_time = time.time() - t0
embeddings = np.vstack(all_emb)
print(f"Done: {inf_time:.1f}s | {len(image_paths)/inf_time:.1f} img/s")


In [ ]:
with h5py.File(OUTPUT_DIR + "/embeddings.hdf5", "w") as f:
    f.create_dataset("embeddings", data=embeddings)
    f.create_dataset("image_ids", data=[s.encode("utf-8") for s in image_ids])
    f.create_dataset("image_paths", data=[s.encode("utf-8") for s in image_paths])

index = faiss.IndexFlatIP(EMBEDDING_DIM)
index.add(embeddings.astype(np.float32))
with open(OUTPUT_DIR + "/faiss_index.pkl", "wb") as f: pickle.dump(index, f)
